# Manual LoRA fine-tuning on Apple Silicon

DSPy's weight optimizers target CUDA-first providers. This self-contained
alternative exports the same classification task to Transformers/PEFT and
trains a small local model with MPS. It is a platform appendix, not a DSPy
optimizer API example.

In [ ]:
from pathlib import Path

_requirements = next(
    (
        candidate / "requirements.txt"
        for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (candidate / "requirements.txt").is_file()
    ),
    None,
)
if _requirements is None:
    raise FileNotFoundError("Could not locate requirements.txt from the current working directory.")
%pip install -r "{_requirements}" -q torch transformers datasets accelerate peft trl

In [ ]:
import os
import random
from pathlib import Path

import dspy
import pandas as pd
from dotenv import load_dotenv

cwd = Path.cwd().resolve()
REPO_ROOT = next(
    (
        candidate
        for candidate in (cwd, *cwd.parents)
        if (candidate / "requirements.txt").is_file()
        and (candidate / "data" / "ai_vs_human200.csv").is_file()
    ),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError(
        f"Could not locate the repository root from {cwd}. "
        "Start Jupyter somewhere inside the cloned repository."
    )
DATA_PATH = REPO_ROOT / "data" / "ai_vs_human200.csv"

load_dotenv(REPO_ROOT / ".env")
NUM_THREADS = int(os.getenv("DSPY_NUM_THREADS", "4"))
TRAIN_LIMIT = int(os.getenv("TRAIN_LIMIT", "32"))
VAL_LIMIT = int(os.getenv("VAL_LIMIT", "12"))
EVAL_LIMIT = int(os.getenv("EVAL_LIMIT", "12"))
print(f"DSPy {dspy.__version__}; local-model workflow")

In [ ]:
class DetectAIText(dspy.Signature):
    """Decide whether the supplied passage was generated by an AI."""

    text: str = dspy.InputField(desc="Passage to classify")
    is_ai: bool = dspy.OutputField(desc="True for AI-generated text; otherwise False")


class AIDetector(dspy.Module):
    def __init__(self):
        super().__init__()
        self.detect = dspy.ChainOfThought(DetectAIText)

    def forward(self, text: str):
        return self.detect(text=text)


def as_bool(value) -> bool:
    if isinstance(value, bool):
        return value
    normalized = str(value).strip().lower()
    if normalized in {"true", "1", "yes"}:
        return True
    if normalized in {"false", "0", "no"}:
        return False
    raise ValueError(f"Cannot interpret {value!r} as a boolean")


def exact_match(example, prediction, trace=None) -> float:
    return float(as_bool(prediction.is_ai) == bool(example.is_ai))


def feedback_metric(example, prediction, trace=None, **kwargs):
    score = exact_match(example, prediction)
    expert_note = getattr(example, "notes", "")
    feedback = (
        "The classification is correct. Preserve the reasoning cues that led to it."
        if score
        else f"The classification is incorrect. The expected is_ai value is {bool(example.is_ai)}. "
             "Focus on concrete stylistic evidence instead of topic or polish alone."
    )
    if expert_note:
        feedback += f" Expert note: {expert_note}"
    return dspy.Prediction(score=score, feedback=feedback)


frame = pd.read_csv(DATA_PATH)
examples = [
    dspy.Example(
        text=row.text,
        is_ai=as_bool(row.is_ai),
        notes="" if pd.isna(row.notes) else str(row.notes),
    ).with_inputs("text")
    for row in frame.itertuples(index=False)
]
random.Random(42).shuffle(examples)

# Deterministic 50/25/25 split; environment variables keep default runs inexpensive.
train_end = len(examples) // 2
val_end = train_end + len(examples) // 4
full_trainset, full_valset, full_testset = (
    examples[:train_end], examples[train_end:val_end], examples[val_end:]
)
trainset = full_trainset[:TRAIN_LIMIT] if TRAIN_LIMIT else full_trainset
valset = full_valset[:VAL_LIMIT] if VAL_LIMIT else full_valset
testset = full_testset[:EVAL_LIMIT] if EVAL_LIMIT else full_testset

def evaluate(program, dataset=testset):
    runner = dspy.Evaluate(
        devset=dataset,
        metric=exact_match,
        num_threads=NUM_THREADS,
        max_errors=max(1, len(dataset)),
        provide_traceback=True,
        failure_score=0.0,
        display_progress=True,
        display_table=5,
    )
    result = runner(program)
    failed = [index for index, (_, prediction, _) in enumerate(result.results) if not hasattr(prediction, "is_ai")]
    if failed:
        raise RuntimeError(f"Evaluation failed for example indexes {failed}; inspect the tracebacks above.")
    return result.score


detector = AIDetector()
print(f"train={len(trainset)}, validation={len(valset)}, test={len(testset)}")

In [ ]:
import torch
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from trl import SFTConfig, SFTTrainer

if not torch.backends.mps.is_available():
    raise RuntimeError("This notebook requires Apple Silicon with PyTorch MPS support.")

LOCAL_MODEL = os.getenv("LOCAL_MODEL", "Qwen/Qwen2.5-0.5B-Instruct")
tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    LOCAL_MODEL,
    torch_dtype=torch.float16,
    device_map={"": "mps"},
)

def training_text(example):
    label = "true" if example.is_ai else "false"
    return (
        "Classify whether the passage was AI generated. Reply only true or false.\n"
        f"Passage: {example.text}\nAnswer: {label}"
    )

dataset = Dataset.from_dict({"text": [training_text(ex) for ex in trainset]})
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
config = SFTConfig(
    output_dir=str(REPO_ROOT / "chapter06" / "runs" / "qwen-ai-detector-mps"),
    dataset_text_field="text",
    max_length=512,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    report_to="none",
    use_mps_device=True,
)
trainer = SFTTrainer(
    model=model,
    args=config,
    train_dataset=dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)
trainer.train()